# Download hub forecasts

Downloads the forecasts submitted to FluSight and the COVID-19 Forecast Hub into `../model-output/<hub>/<model>/`.
No GitHub token is needed: one anonymous GitHub API call per hub lists the files (limit: 60 calls per hour), and the files are then downloaded from `raw.githubusercontent.com`.

Hub reference dates are shifted back by 7 days so that they match the reference dates of the TSFM forecasts.

In [ ]:
import os
import time
from io import BytesIO

import pandas as pd
import requests

HUB_REPOS = {
    "FluSight": ("cdcepi", "FluSight-forecast-hub"),
    "CovidHub": ("CDCgov", "covid19-forecast-hub"),
}

In [ ]:
# Hub reference dates of the 2025/26 evaluation rounds (TSFM convention 2025-11-22 ... 2026-05-23, i.e. hub date - 7 days)
START, END = "2025-11-29", "2026-05-30"
# csv and parquet submissions (in this window, UMass-trends_ensemble submits parquet files)
EXTENSIONS = (".csv", ".parquet")

In [ ]:
def list_model_output_files(owner, repo, start, end, extensions, branch="main"):
    """All model-output files of a hub repository with hub reference date in [start, end]."""
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    tree = r.json()
    if tree.get("truncated"):
        raise RuntimeError(f"File list of {repo} is truncated")
    files = []
    for item in tree["tree"]:
        parts = item["path"].split("/")
        if item["type"] != "blob" or len(parts) != 3 or parts[0] != "model-output":
            continue
        model, filename = parts[1], parts[2]
        if filename.endswith(extensions) and start <= filename[:10] <= end:
            files.append((model, filename, item["path"]))
    return files


def download_forecast(owner, repo, path, branch="main"):
    r = requests.get(f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path}", timeout=120)
    r.raise_for_status()
    return pd.read_parquet(BytesIO(r.content)) if path.endswith(".parquet") else pd.read_csv(BytesIO(r.content))


def shift_reference_date(df):
    """Hub reference date - 7 days, as for the TSFM forecasts."""
    df["reference_date"] = (pd.to_datetime(df["reference_date"]) - pd.Timedelta(days=7)).dt.strftime("%Y-%m-%d")
    return df

In [ ]:
for hub, (owner, repo) in HUB_REPOS.items():
    files = list_model_output_files(owner, repo, START, END, EXTENSIONS)
    print(f"{hub}: {len(files)} files from {len({m for m, _, _ in files})} models")
    for model, filename, path in files:
        df = shift_reference_date(download_forecast(owner, repo, path))
        local_dir = f"../model-output/{hub}/{model}"
        os.makedirs(local_dir, exist_ok=True)
        df.to_csv(os.path.join(local_dir, f"{df['reference_date'].iloc[0]}-{model}.csv"), index=False)
        time.sleep(0.2)